In [0]:
%sql
SELECT * FROM catalog.silver.fact_visits LIMIT 10;

attending_physician_id,diagnosis_code,hospital_id,patient_id,visit_id,admission_date,discharge_date,length_of_stay_days,total_cost_php,is_30day_readmission,first_name,last_name,gender,dob,patient_city,patient_assigned_branch,patient_name_hash,hospital_name,hospital_city,network_region_cluster,hospital_class,bed_count,diagnosis_desc,diagnosis_category,readmission_risk_level,specialization,load_timestamp,days_since_last_discharge
null,C34,H001,P0179,V00236,2022-12-22,2022-12-28,6,149000,false,Christian,Aquino,M,1972-04-14,Iligan,H007,47a2de2a99379f79c9a9a5b766c7d7ea5b75e535d9337baae70266858e5c1efc,MediLuzon General Hospital - Manila Flagship,Manila,NCR Cluster,Tertiary,350,Lung Cancer,Oncology,High,null,2026-05-24T14:24:56.656Z,null
DR031,A41,H007,P0179,V00376,2023-01-11,2023-01-24,13,101000,true,Christian,Aquino,M,1972-04-14,Iligan,H007,47a2de2a99379f79c9a9a5b766c7d7ea5b75e535d9337baae70266858e5c1efc,MediLuzon Community Hospital - Iloilo,Iloilo City,Visayas Cluster,Secondary,130,Sepsis,Critical Care,High,Cardiology,2026-05-24T14:24:56.656Z,14
DR046,J18,H002,P0179,V00558,2023-02-14,2023-02-22,8,16000,true,Christian,Aquino,M,1972-04-14,Iligan,H007,47a2de2a99379f79c9a9a5b766c7d7ea5b75e535d9337baae70266858e5c1efc,MediLuzon General Hospital - Quezon City,Quezon City,NCR Cluster,Tertiary,280,Pneumonia,Respiratory,Medium,Nephrology,2026-05-24T14:24:56.656Z,21
DR049,A15,H001,P0179,V02123,2023-05-22,2023-05-24,2,50000,false,Christian,Aquino,M,1972-04-14,Iligan,H007,47a2de2a99379f79c9a9a5b766c7d7ea5b75e535d9337baae70266858e5c1efc,MediLuzon General Hospital - Manila Flagship,Manila,NCR Cluster,Tertiary,350,Pulmonary Tuberculosis,Respiratory,Medium,Oncology,2026-05-24T14:24:56.656Z,89
DR033,I50,H001,P0179,V00619,2023-06-07,2023-06-12,5,54000,true,Christian,Aquino,M,1972-04-14,Iligan,H007,47a2de2a99379f79c9a9a5b766c7d7ea5b75e535d9337baae70266858e5c1efc,MediLuzon General Hospital - Manila Flagship,Manila,NCR Cluster,Tertiary,350,Heart Failure,Cardiovascular,High,Obstetrics,2026-05-24T14:24:56.656Z,14
DR037,C50,H009,P0179,V00929,2023-07-09,2023-07-14,5,40000,true,Christian,Aquino,M,1972-04-14,Iligan,H007,47a2de2a99379f79c9a9a5b766c7d7ea5b75e535d9337baae70266858e5c1efc,MediLuzon Community Hospital - Cagayan de Oro,Cagayan de Oro,Mindanao Cluster,Secondary,110,Breast Cancer,Oncology,High,Infectious Disease,2026-05-24T14:24:56.656Z,27
DR048,Z49,H005,P0179,V02549,2023-07-12,2023-07-14,2,17000,false,Christian,Aquino,M,1972-04-14,Iligan,H007,47a2de2a99379f79c9a9a5b766c7d7ea5b75e535d9337baae70266858e5c1efc,MediLuzon Community Hospital - Pampanga,"San Fernando, Pampanga",Luzon Cluster,Secondary,150,Dialysis Care Encounter,Procedure,High,Infectious Disease,2026-05-24T14:24:56.656Z,-2
DR008,J18,H005,P0179,V02100,2024-12-12,2024-12-18,6,24000,false,Christian,Aquino,M,1972-04-14,Iligan,H007,47a2de2a99379f79c9a9a5b766c7d7ea5b75e535d9337baae70266858e5c1efc,MediLuzon Community Hospital - Pampanga,"San Fernando, Pampanga",Luzon Cluster,Secondary,150,Pneumonia,Respiratory,Medium,Gastroenterology,2026-05-24T14:24:56.656Z,517
DR020,A15,H005,P0243,V01294,2023-03-28,2023-03-30,2,38000,false,Veronica,Morales,F,1964-08-22,Cebu City,H004,6615e4dc33b6be458341cd540677c11626de0f519a9d910e01d75ec1d29ea908,MediLuzon Community Hospital - Pampanga,"San Fernando, Pampanga",Luzon Cluster,Secondary,150,Pulmonary Tuberculosis,Respiratory,Medium,Neurology,2026-05-24T14:24:56.656Z,null
DR046,Z51,H003,P0243,V01822,2023-04-08,2023-04-09,1,76000,true,Veronica,Morales,F,1964-08-22,Cebu City,H004,6615e4dc33b6be458341cd540677c11626de0f519a9d910e01d75ec1d29ea908,MediLuzon Specialist Center - Makati,Makati,NCR Cluster,Specialty,180,Chemotherapy / Radiotherapy,Procedure,High,Nephrology,2026-05-24T14:24:56.656Z,9


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
df = spark.read.table("catalog.silver.fact_visits")

In [0]:
# Readmission analysis by hospital and diagnosis
gold_df = (
    df.groupBy(
        "hospital_id",
        "hospital_name",
        "diagnosis_desc",
    )
    .agg(
        count("*").alias("total_visits"),
        sum(col("is_30day_readmission").cast("int")).alias("total_readmissions"),
        round(sum(col("is_30day_readmission").cast("int")) / count("*"), 3).alias("readmission_rate"),
        sum("total_cost_php").alias("total_cost"),
        round(avg("total_cost_php")).alias("avg_cost")
    )
    .withColumn("gold_load_timestamp", current_timestamp()
    )
)

In [0]:
display(gold_df.orderBy("readmission_rate", ascending=False).head(10))

hospital_id,hospital_name,diagnosis_desc,total_visits,total_readmissions,readmission_rate,total_cost,avg_cost,gold_load_timestamp
H010,MediLuzon Specialist Center - Bonifacio Global City,Lung Cancer,2,2,1.0,163000.0,81500.0,2026-05-28T08:30:02.722Z
H008,MediLuzon General Hospital - Davao,Lung Cancer,2,2,1.0,186000.0,93000.0,2026-05-28T08:30:02.722Z
H010,MediLuzon Specialist Center - Bonifacio Global City,Epilepsy,1,1,1.0,42000.0,42000.0,2026-05-28T08:30:02.722Z
H005,MediLuzon Community Hospital - Pampanga,Alcoholic Liver Disease,1,1,1.0,49000.0,49000.0,2026-05-28T08:30:02.722Z
H009,MediLuzon Community Hospital - Cagayan de Oro,Chronic Kidney Disease (CKD),15,13,0.867,854000.0,56933.0,2026-05-28T08:30:02.722Z
H010,MediLuzon Specialist Center - Bonifacio Global City,Dengue Hemorrhagic Fever,7,6,0.857,262000.0,37429.0,2026-05-28T08:30:02.722Z
H002,MediLuzon General Hospital - Quezon City,Lung Cancer,6,5,0.833,514000.0,85667.0,2026-05-28T08:30:02.722Z
H002,MediLuzon General Hospital - Quezon City,Chronic Kidney Disease (CKD),22,18,0.818,1067000.0,48500.0,2026-05-28T08:30:02.722Z
H008,MediLuzon General Hospital - Davao,Chemotherapy / Radiotherapy,10,8,0.8,940000.0,94000.0,2026-05-28T08:30:02.722Z
H009,MediLuzon Community Hospital - Cagayan de Oro,Dialysis Care Encounter,10,8,0.8,122000.0,12200.0,2026-05-28T08:30:02.722Z


In [0]:
gold_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("catalog.gold.hospital_diagnosis_kpi")

In [0]:
# Hospital Region KPI
gold_region_df = (
    df.groupBy(
        "network_region_cluster"
    )
    .agg(
        count("*").alias("total_visits"),
        sum(col("is_30day_readmission").cast("int")).alias("total_readmissions"),
        round(
            sum(col("is_30day_readmission").cast("int")) / count("*"),
            3
        ).alias("readmission_rate"),
        sum("total_cost_php").alias("total_cost"),
        round(avg("total_cost_php"), 2).alias("avg_cost")
    )
    .withColumn("gold_load_timestamp", current_timestamp()
    )
)

In [0]:
display(gold_region_df.orderBy("readmission_rate", ascending=False))

network_region_cluster,total_visits,total_readmissions,readmission_rate,total_cost,avg_cost,gold_load_timestamp
Mindanao Cluster,563,228,0.405,3.2515E7,57753.11,2026-05-25T14:37:29.724Z
NCR Cluster,1188,460,0.387,6.8644E7,57781.14,2026-05-25T14:37:29.724Z
Visayas Cluster,618,230,0.372,3.6262E7,58676.38,2026-05-25T14:37:29.724Z
Luzon Cluster,631,229,0.363,3.5224E7,55822.5,2026-05-25T14:37:29.724Z


In [0]:
gold_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("catalog.gold.hospital_region_kpi")

In [0]:
# Diagnosis KPI 
gold_diagnosis_df = (
    df.groupBy(
        "diagnosis_desc",
        "diagnosis_code",
        "diagnosis_category"
    )
    .agg(
        count("*").alias("total_cases"),
        sum(col("is_30day_readmission").cast("int")).alias("total_readmissions"),
        round(avg(col("is_30day_readmission").cast("int")), 3).alias("readmission_rate"),
        round(avg(col("total_cost_php")),2).alias("avg_cost"),
        round(avg(col("length_of_stay_days")), 2).alias("avg_length_of_stay")
    )
    .withColumn("gold_load_timestamp", current_timestamp()
    )
)

In [0]:
display(gold_diagnosis_df.orderBy("readmission_rate", ascending=False).head(15))

diagnosis_desc,diagnosis_code,diagnosis_category,total_cases,total_readmissions,readmission_rate,avg_cost,avg_length_of_stay,gold_load_timestamp
Chronic Kidney Disease (CKD),N18,Renal,175,119,0.68,52880.0,5.26,2026-05-28T08:31:25.583Z
Chemotherapy / Radiotherapy,Z51,Procedure,99,57,0.576,83272.73,2.49,2026-05-28T08:31:25.583Z
Dialysis Care Encounter,Z49,Procedure,99,53,0.535,30010.1,2.32,2026-05-28T08:31:25.583Z
Lung Cancer,C34,Oncology,52,27,0.519,85615.38,5.73,2026-05-28T08:31:25.583Z
Heart Failure,I50,Cardiovascular,197,96,0.487,68116.75,6.93,2026-05-28T08:31:25.583Z
Sepsis,A41,Critical Care,121,53,0.438,114925.62,9.58,2026-05-28T08:31:25.583Z
Colon Cancer,C18,Oncology,40,17,0.425,95300.0,5.67,2026-05-28T08:31:25.583Z
Intracerebral Hemorrhage,I61,Cardiovascular,66,28,0.424,98378.79,9.55,2026-05-28T08:31:25.583Z
Acute Kidney Failure,N17,Renal,100,42,0.42,51520.0,4.5,2026-05-28T08:31:25.583Z
Acute Myocardial Infarction,I21,Cardiovascular,125,50,0.4,105152.0,7.2,2026-05-28T08:31:25.583Z


In [0]:
gold_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("catalog.gold.diagnosis_kpi")

In [0]:
# Hospitals KPI
gold_hospitals_df = (
    df.groupBy(
        "hospital_id",
        "hospital_name",
        "hospital_class"
    )
    .agg(
        count("*").alias("total_visits"),
        countDistinct("patient_id").alias("total_patients"),
        sum(col("is_30day_readmission").cast("int")).alias("total_readmissions"),
        round(avg(col("is_30day_readmission").cast("int")), 3).alias("readmission_rate"),
        round(avg(col("length_of_stay_days")), 2).alias("avg_length_of_stay"),
        sum("total_cost_php").alias("total_cost"),
        round(avg("total_cost_php"), 2).alias("avg_cost_per_visit")
    )
    .withColumn("gold_load_timestamp", current_timestamp()
    )
)

In [0]:
display(gold_hospitals_df.orderBy("readmission_rate", ascending=False))

hospital_id,hospital_name,hospital_class,total_visits,total_patients,total_readmissions,readmission_rate,avg_length_of_stay,total_cost,avg_cost_per_visit,gold_load_timestamp
H008,MediLuzon General Hospital - Davao,Tertiary,270,204,112,0.415,5.16,1.5727E7,58248.15,2026-05-28T08:31:44.761Z
H009,MediLuzon Community Hospital - Cagayan de Oro,Secondary,293,220,116,0.396,5.1,1.6788E7,57296.93,2026-05-28T08:31:44.761Z
H001,MediLuzon General Hospital - Manila Flagship,Tertiary,321,225,126,0.393,5.21,1.8217E7,56750.78,2026-05-28T08:31:44.761Z
H002,MediLuzon General Hospital - Quezon City,Tertiary,291,222,114,0.392,5.27,1.6588E7,57003.44,2026-05-28T08:31:44.761Z
H010,MediLuzon Specialist Center - Bonifacio Global City,Specialty,294,221,115,0.391,5.44,1.6682E7,56741.5,2026-05-28T08:31:44.761Z
H007,MediLuzon Community Hospital - Iloilo,Secondary,304,226,116,0.382,5.25,1.7075E7,56167.76,2026-05-28T08:31:44.761Z
H004,MediLuzon Community Hospital - Batangas,Secondary,312,230,117,0.375,5.34,1.6991E7,54458.33,2026-05-28T08:31:44.761Z
H003,MediLuzon Specialist Center - Makati,Specialty,282,211,105,0.372,5.26,1.7157E7,60840.43,2026-05-28T08:31:44.761Z
H006,MediLuzon General Hospital - Cebu,Tertiary,314,228,114,0.363,5.28,1.9187E7,61105.1,2026-05-28T08:31:44.761Z
H005,MediLuzon Community Hospital - Pampanga,Secondary,319,236,112,0.351,5.18,1.8233E7,57156.74,2026-05-28T08:31:44.761Z


In [0]:
gold_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("catalog.gold.hospitals_kpi")

In [0]:
# Physicians KPI
gold_physicians_df = (
    df.groupBy(
        "attending_physician_id",
        "specialization"
    )
    .agg(
        count("*").alias("total_visits"),
        countDistinct("patient_id").alias("total_patients"),
        sum(col("is_30day_readmission").cast("int")).alias("total_readmissions"),
        round(avg(col("is_30day_readmission").cast("int")), 3).alias("readmission_rate"),
        round(avg(col("length_of_stay_days")), 2).alias("avg_length_of_stay"),
        round(avg("total_cost_php"), 2).alias("avg_cost_per_patient")
    )
    .withColumn("gold_load_timestamp", current_timestamp()
    )
)

In [0]:
display(gold_physicians_df.orderBy("readmission_rate", ascending=False).head(10))

attending_physician_id,specialization,total_visits,total_patients,total_readmissions,readmission_rate,avg_length_of_stay,avg_cost_per_patient,gold_load_timestamp
DR011,Oncology,69,66,39,0.565,5.68,69826.09,2026-05-28T08:32:06.983Z
DR012,Pulmonology,46,43,24,0.522,5.17,49500.0,2026-05-28T08:32:06.983Z
DR025,Obstetrics,51,51,26,0.51,5.67,54941.18,2026-05-28T08:32:06.983Z
DR028,Neurology,53,52,26,0.491,5.12,68641.51,2026-05-28T08:32:06.983Z
DR043,Neurology,59,55,28,0.475,5.64,59338.98,2026-05-28T08:32:06.983Z
DR050,Oncology,55,54,26,0.473,5.4,62218.18,2026-05-28T08:32:06.983Z
DR004,Endocrinology,55,51,26,0.473,4.94,44672.73,2026-05-28T08:32:06.983Z
DR049,Oncology,70,65,33,0.471,5.07,59300.0,2026-05-28T08:32:06.983Z
DR038,Endocrinology,60,56,28,0.467,5.34,51966.67,2026-05-28T08:32:06.983Z
DR044,Gastroenterology,51,48,23,0.451,5.49,59823.53,2026-05-28T08:32:06.983Z


In [0]:
gold_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("catalog.gold.physicians_kpi")

In [0]:
# Cost KPI
gold_cost_df = (
    df.groupBy(
        "hospital_id",
        "hospital_name",
        "diagnosis_desc"
    )
    .agg(
        sum("total_cost_php").alias("total_cost"),
        round(avg("total_cost_php"), 2).alias("avg_cost"),
        round(avg("length_of_stay_days"), 2).alias("avg_length_of_stay"),
        sum(col("is_30day_readmission").cast("int")).alias("total_readmissions")
    )
    .withColumn("gold_load_timestamp", current_timestamp()
    )
)

In [0]:
display(gold_cost_df.head(10))

hospital_id,hospital_name,diagnosis_desc,total_cost,avg_cost,avg_length_of_stay,total_readmissions,gold_load_timestamp
H001,MediLuzon General Hospital - Manila Flagship,Lung Cancer,727000.0,103857.14,6.0,2,2026-05-28T08:32:23.070Z
H007,MediLuzon Community Hospital - Iloilo,Sepsis,1414000.0,128545.45,9.0,8,2026-05-28T08:32:23.070Z
H002,MediLuzon General Hospital - Quezon City,Pneumonia,526000.0,37571.43,5.36,5,2026-05-28T08:32:23.070Z
H001,MediLuzon General Hospital - Manila Flagship,Pulmonary Tuberculosis,558000.0,50727.27,4.36,3,2026-05-28T08:32:23.070Z
H001,MediLuzon General Hospital - Manila Flagship,Heart Failure,1256000.0,73882.35,7.0,9,2026-05-28T08:32:23.070Z
H009,MediLuzon Community Hospital - Cagayan de Oro,Breast Cancer,203000.0,40600.0,4.0,2,2026-05-28T08:32:23.070Z
H005,MediLuzon Community Hospital - Pampanga,Dialysis Care Encounter,226000.0,32285.71,2.71,3,2026-05-28T08:32:23.070Z
H005,MediLuzon Community Hospital - Pampanga,Pneumonia,1022000.0,44434.78,5.82,7,2026-05-28T08:32:23.070Z
H005,MediLuzon Community Hospital - Pampanga,Pulmonary Tuberculosis,393000.0,43666.67,3.89,3,2026-05-28T08:32:23.070Z
H003,MediLuzon Specialist Center - Makati,Chemotherapy / Radiotherapy,943000.0,72538.46,2.0,7,2026-05-28T08:32:23.070Z


In [0]:
gold_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("catalog.gold.cost_kpi")